# LLaVA 图像描述 —— 面向视障用户的场景解说（第 1 周）

## 练习目标（理念）

用本地 **Ollama + LLaVA** 多模态模型：上传图片 → 生成对视障用户友好的场景描述。

如果你也对这个方向感兴趣，欢迎在社区贡献想法或代码。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 调用本地模型 | `ollama.generate(...)` |
| 多模态输入 | `images=` 传入 base64 图片列表 |
| 流式输出 `stream=True` | 边生成边 `print` |
| Prompt 设计 | system 定「怎么描述」，user 可追加个人问题 |

## 怎么跑

1. 本机已安装并启动 Ollama，且已拉取 `llava:7b-v1.6`
2. 从上到下依次运行单元格；`call_llava()` 会交互式询问图片路径与补充 prompt
3. 图片路径按你本机实际情况填写（Windows / Linux 均可）


## 项目背景（为什么做这件事）

对视障人士来说，日常生活中有许多我们习以为常的障碍。盲文、导盲犬等工具能提供部分支持，但还远不足以覆盖「理解眼前场景」这类需求。

全球有超过 **4330 万** 盲人；本项目希望用**实时、情境准确**的图像描述，帮助他们：

- 更好地感知周围环境
- 减少孤立感、增强自主感
- 不仅是「导航辅助」，更是与世界建立连接的桥梁

技术路线：用 **LLaVA** 看图说话，并用关怀、清晰、少术语的语言输出描述。


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入 ollama：用 Python SDK 调用本机 Ollama（含多模态 generate）
import ollama
# 导入标准库 base64：把图片二进制编码成文本，便于塞进 API 请求
import base64
# 导入标准库 os：做路径规范化、判断文件是否存在等
import os


In [ ]:
# ========== 工具函数：把本地图片读成 base64 字符串 ==========

def encode_image(image_path):
    # 以二进制只读方式打开图片文件
    with open(image_path, 'rb') as f:
        # b64encode → bytes；再 decode 成 utf-8 字符串，供 ollama.generate(images=...) 使用
        return base64.b64encode(f.read()).decode('utf-8')


In [ ]:
# ==========（可选）本地自测：编码一张图并预览 base64 前缀 ==========
# 下面三行默认注释掉：解开后换成你自己的图片路径即可试跑 encode_image
# image_path = r"C:\Users\LAKSHYA\OneDrive\Pictures\Camera Roll\WIN_20250614_02_46_47_Pro.jpg"
# image_base64 = encode_image(image_path)
# print(image_base64[:100])


In [ ]:
# ========== 全局状态：存放已编码的图片（base64 字符串列表）==========
# 后续 put_image() 会往这里 append；call_llava() 每次开始会 clear
image_list = []


In [ ]:
# ========== 交互：让用户输入图片路径，编码后放进 image_list ==========

def put_image():
    # 声明使用模块级全局列表（同一会话里跨单元格共享）
    global image_list
    # input：在笔记本里弹出交互提示；strip 去掉首尾空白
    user_input_image = input("Enter image path or press enter to skip: ").strip()

    # 空输入：本次不插图，直接返回当前列表
    if not user_input_image:
        print("No image inserted")
        return image_list

    # normpath：统一路径分隔符（Windows \ 与 POSIX / 混用时更稳）
    image_path = os.path.normpath(user_input_image)

    # 路径不存在：提示后递归再问一次（原逻辑保留）
    if not os.path.exists(image_path):
        print("Image path not found! Try again or enter to leave blank")
        return put_image()  # Continue to allow more inputs





    # 读文件 → base64，追加到全局列表
    image_base64 = encode_image(image_path)
    image_list.append(image_base64)

    # Detect file extension for MIME type
    # 下面是预留的 MIME 拼接思路（data URL），当前未启用
    # ext = os.path.splitext(image_path)[-1].lower()
    # mime_type = 'image/jpeg' if ext in ['.jpg', '.jpeg'] else 'image/png'  # Extend if needed


    return image_list

    # return f"data:{mime_type};base64,{image_base64[:100]}"


In [ ]:
# ========== System Prompt：告诉 LLaVA「怎么为视障用户描述画面」==========
# 发给模型的英文指令保留原文：翻译会改变回答风格/行为
prompt=  ("System prompt: (You are a compassionate and intelligent visual assistant designed to help people who are blind or visually impaired. "
    "Your job is to look at an image and describe it in a way that helps the user understand the scene clearly. "
    "Use simple, descriptive language and avoid technical terms. Describe what is happening in the image, people's body language, clothing, facial expressions, objects, and surroundings. "
    "Be vivid and precise, as if you are painting a picture with words. "
    "Also, take into account any personal instructions or questions provided by the user—such as describing a specific person, activity, or object. "
    "If the user includes a specific prompt, prioritize that in your description.)")


In [ ]:
# ========== 交互：追加用户侧补充问题到 prompt ==========

def put_prompt():
    # 使用并可能改写全局 prompt 字符串
    global prompt
    # 让用户输入个性化问题（例如「桌上有没有杯子？」）
    user_input = input("Put new prompt: ")
    # 空输入则递归重问（原逻辑保留）
    if not user_input:
        print("please enter a prompt")
        return put_prompt()
    # 把用户话追加到对话文本里，供本次 generate 使用
    prompt += "\nUser: " + user_input
    return prompt


In [ ]:
# ========== 主流程：选图 → 补 prompt → 流式调用 LLaVA ==========

def image_description():
    global prompt

    # 先收集至少一张图（也可能用户跳过）
    put_image()
    # 列表为空：无法做多模态描述，直接返回提示字符串
    if not image_list:
        return "No images available. Skipping..."

    # 拿到（可能已追加用户话的）完整 prompt
    user_prompt = put_prompt()
    # full_answer：把流式碎片拼成完整回答，便于事后写回对话历史
    full_answer = ""

    # ollama.generate：本地推理；images 传 base64 列表；stream=True 边生成边收
    for chunk in ollama.generate(
        model='llava:7b-v1.6',
        prompt=user_prompt,
        images=image_list,
        stream=True
    ):
        # 每个 chunk 是 dict；文本增量在 "response" 字段
        content = chunk.get("response", "")
        # end="" + flush=True：不换行缓冲，实现「打字机」式直播
        print("\n\n Final Answer:",content, end="", flush=True)  # Live stream to console
        full_answer += content

    # 把本轮 User/Assistant 写回全局 prompt，便于多轮上下文（原逻辑保留）
    prompt += "\nUser: " + user_prompt + "\nAssistant: " + full_answer
    return full_answer


In [ ]:
# ========== 入口封装：清空图片列表，连续跑最多 5 轮描述 ==========

def call_llava():
    # 每轮会话前清空，避免上一轮图片残留
    image_list.clear()
    # 固定迭代 5 次（原逻辑）；中间可跳过插图
    for i in range(5):
        print(f"\n Iteration {i+1}")
        # 跑一轮：交互选图 + 提问 + 流式生成
        answer = image_description()
        # 再打印一次完整答案（与流式过程中的打印互补）
        print("\n\n Final Answer:", answer)


In [ ]:
# ========== 运行：启动交互式 LLaVA 图像描述 ==========
# 需本机 Ollama 已拉取 llava:7b-v1.6；按提示输入图片路径与补充问题
call_llava()
